# Brand Shopping Agent Demo: From Static PDPs to Personalized Economics

This notebook demonstrates the concept shown in the image: the same product can lead to different economic outcomes depending on user goals.

We model two users:
- User A: optimize for speed and protection
- User B: optimize for total cost and cash flow

The demo includes guardrails, candidate offer generation, scoring, and an explainable recommendation output.

## Business Framing

Traditional PDPs show static content: one price, one set of defaults, and one generic CTA.

A shopping agent turns this into personalized economics by combining:
- Product constraints (price floor, margin guardrails)
- Operational context (shipping SLAs, inventory)
- User preferences (speed, risk, monthly cash flow)

Result: each user gets the best offer for their priorities, not a one-size-fits-all bundle.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List
from pprint import pprint

@dataclass
class Product:
    sku: str
    name: str
    list_price: float
    min_price: float

@dataclass
class ShippingOption:
    name: str
    days: int
    cost: float

@dataclass
class WarrantyOption:
    years: int
    cost: float

@dataclass
class FinancingOption:
    months: int
    apr: float

@dataclass
class UserProfile:
    name: str
    speed_weight: float
    protection_weight: float
    total_cost_weight: float
    monthly_payment_weight: float
    max_monthly_payment: float

In [ ]:
product = Product(
    sku='HD-CAM-X',
    name='HD Digital Camera Model X',
    list_price=699.0,
    min_price=659.0,
)

shipping_options = [
    ShippingOption(name='Express', days=1, cost=19.0),
    ShippingOption(name='Standard', days=3, cost=0.0),
]

warranty_options = [
    WarrantyOption(years=0, cost=0.0),
    WarrantyOption(years=2, cost=79.0),
]

financing_options = [
    FinancingOption(months=1, apr=0.0),
    FinancingOption(months=12, apr=0.0),
]

# Policy guardrails: discount and installment limits
max_discount = 0.06

users = [
    UserProfile(
        name='User A',
        speed_weight=0.45,
        protection_weight=0.35,
        total_cost_weight=0.10,
        monthly_payment_weight=0.10,
        max_monthly_payment=800.0,
    ),
    UserProfile(
        name='User B',
        speed_weight=0.05,
        protection_weight=0.10,
        total_cost_weight=0.45,
        monthly_payment_weight=0.40,
        max_monthly_payment=65.0,
    ),
]

print('Product and profiles loaded.')

In [ ]:
def monthly_payment(total_amount: float, months: int) -> float:
    return round(total_amount / months, 2)

def candidate_price(base_price: float, user: UserProfile) -> float:
    # Small policy-safe personalization: price-sensitive users may get max allowed discount
    if user.total_cost_weight >= 0.4:
        discounted = base_price * (1.0 - max_discount)
        return round(max(discounted, product.min_price), 2)
    return base_price

def normalize_speed(days: int) -> float:
    # 1 day is best, 5+ days treated as poor
    return max(0.0, min(1.0, (5 - days) / 4))

def normalize_protection(years: int) -> float:
    return max(0.0, min(1.0, years / 2))

def normalize_total_cost(total_cost: float) -> float:
    # Lower total cost gets higher score
    worst = product.list_price + 150
    best = product.min_price
    return max(0.0, min(1.0, (worst - total_cost) / (worst - best)))

def normalize_monthly(payment: float) -> float:
    # Lower payment is better
    worst = 800.0
    return max(0.0, min(1.0, (worst - payment) / worst))

def score_offer(user: UserProfile, offer: Dict) -> float:
    s_speed = normalize_speed(offer['shipping_days'])
    s_prot = normalize_protection(offer['warranty_years'])
    s_cost = normalize_total_cost(offer['total_cost'])
    s_monthly = normalize_monthly(offer['monthly_payment'])

    score = (
        user.speed_weight * s_speed
        + user.protection_weight * s_prot
        + user.total_cost_weight * s_cost
        + user.monthly_payment_weight * s_monthly
    )
    return round(score, 4)

def build_offers(user: UserProfile) -> List[Dict]:
    offers = []
    unit_price = candidate_price(product.list_price, user)

    for ship in shipping_options:
        for warranty in warranty_options:
            for financing in financing_options:
                total = round(unit_price + ship.cost + warranty.cost, 2)
                monthly = monthly_payment(total, financing.months)

                if monthly > user.max_monthly_payment:
                    continue

                offer = {
                    'user': user.name,
                    'price': unit_price,
                    'shipping_name': ship.name,
                    'shipping_days': ship.days,
                    'warranty_years': warranty.years,
                    'financing_months': financing.months,
                    'total_cost': total,
                    'monthly_payment': monthly,
                }
                offer['score'] = score_offer(user, offer)
                offers.append(offer)

    offers.sort(key=lambda x: x['score'], reverse=True)
    return offers

In [ ]:
def explain(offer: Dict) -> str:
    parts = [
        f"Price ${offer['price']:.2f}",
        f"{offer['shipping_name']} shipping ({offer['shipping_days']} day)",
        f"{offer['warranty_years']} year warranty",
        f"{offer['financing_months']} month plan",
        f"Total ${offer['total_cost']:.2f}",
        f"Monthly ${offer['monthly_payment']:.2f}",
    ]
    return ' | '.join(parts)

top_offers = {}
for user in users:
    offers = build_offers(user)
    top = offers[0] if offers else None
    top_offers[user.name] = top

print('Top recommendation by user:')
for user_name, offer in top_offers.items():
    print(f'\n{user_name}')
    if offer is None:
        print('  No valid offer under guardrails.')
    else:
        print(' ', explain(offer))
        print(' ', f"Decision score: {offer['score']}")

In [ ]:
def compact_row(user_name: str, offer: Dict) -> Dict:
    return {
        'user': user_name,
        'price': offer['price'],
        'ship': f"{offer['shipping_name']} ({offer['shipping_days']}d)",
        'warranty_years': offer['warranty_years'],
        'financing_months': offer['financing_months'],
        'total_cost': offer['total_cost'],
        'monthly_payment': offer['monthly_payment'],
        'score': offer['score'],
    }

summary = [compact_row(k, v) for k, v in top_offers.items() if v]
pprint(summary)

## How This Maps to the Slide

- **Same product**: one camera SKU
- **User A outcome**: tends to choose express shipping + stronger protection
- **User B outcome**: tends to choose lower total cost and lower monthly payment
- **Real-time personalization**: the scoring function responds to user weights + guardrails

For a live demo, connect this logic to:
1. Real inventory and shipping SLA feeds
2. Dynamic promotions service
3. Credit/financing eligibility checks
4. Explainability text for customer trust